<a href="https://colab.research.google.com/github/Evan20230427/colab-rag-qa/blob/main/RAG_QA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab LLM + RAG 本地知識問答系統
## 基於 Ollama + ChromaDB + Streamlit，部署於 Google Colab

---

### 技術架構
| 元件 | 工具 | 說明 |
|------|------|------|
| 嵌入模型 | mxbai-embed-large | 將文字轉為向量，用於語義搜尋 |
| 對話 LLM | ycchen/breeze-7b-instruct-v1_0 | 聯發科繁體中文優化語言模型 |
| 推論引擎 | Ollama | 本地 LLM 執行環境 |
| 向量資料庫 | ChromaDB | 儲存並查詢知識庫嵌入向量 |
| Web UI | Streamlit | 互動式問答介面 |
| 對外隧道 | pyngrok | 將 Colab 內部埠號公開為 HTTPS URL |

### 使用前請確認
1. **Runtime 已設為 T4 GPU**：Runtime -> Change runtime type -> T4 GPU
2. **取得 Ngrok authtoken**：
   - 前往 https://dashboard.ngrok.com/get-started/your-authtoken
   - 以 Google 帳號 (evanchen628@gmail.com) 登入
   - 複製 authtoken 填入 Cell 2 的 NGROK_AUTHTOKEN
3. **依序執行所有 Cell**（Cell 1 -> Cell 9）

### 預計等待時間
- Cell 3（套件安裝）：約 2 分鐘
- Cell 4（Ollama 安裝）：約 1 分鐘
- Cell 5（模型下載）：約 5~15 分鐘（模型總大小約 5GB）
- Cell 9（啟動服務）：約 30 秒

### Google Drive 整合說明（帳號：evanchen628@gmail.com）
- Cell 7 掛載 Drive 時，請以 evanchen628@gmail.com 授權
- 知識庫（QA50.xlsx）與向量索引（ChromaDB）將儲存於：我的雲端硬碟 > RAG_QA/
- 重啟 Colab 後，從 Cell 4 開始執行即可，知識庫索引已持久保存無需重建

In [ ]:
# ============================================================
# Cell 2｜使用者設定區
# 只需修改此 Cell，其他 Cell 無需改動
# ============================================================

from google.colab import userdata

# 安全地從 Colab 金鑰管理器 (Secrets) 讀取 Ngrok authtoken
# 請確保已在左側「🔑 Secrets」面板中新增名稱為 NGROK_AUTHTOKEN 的金鑰並勾選存取權
try:
    NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
except userdata.NotebookAccessError:
    raise RuntimeError("未授權存取金鑰！請點擊左側 🔑 圖示，並勾選 NGROK_AUTHTOKEN 的 'Notebook access' 權限。")
except Exception:
    # 若無設定金鑰，給予直覺的提示並使用預設空字串
    NGROK_AUTHTOKEN = ""

# Google Drive 整合設定
# True  = 知識庫與向量索引持久保存於 Drive（推薦：重啟後無需重建索引）
# False = 使用記憶體模式，每次 Session 結束後資料消失
USE_GOOGLE_DRIVE = True

# Google Drive 中 QA Excel 檔案的路徑
GDRIVE_QA_PATH = "/content/drive/MyDrive/ChatbotRAG/第二層面試FAQ題庫整併.xlsx"

# ChromaDB 向量索引的持久化目錄
CHROMA_PERSIST_DIR = "/content/drive/MyDrive/ChatbotRAG/chroma_db"

# 文字嵌入模型（用於將文字轉為語義向量）
EMBEDDING_MODEL = "mxbai-embed-large"

# 對話語言模型（繁體中文優化版，聯發科 Breeze-7B）
LLM_MODEL = "ycchen/breeze-7b-instruct-v1_0"

# 設定驗證
assert NGROK_AUTHTOKEN, \
    "未偵測到 Ngrok authtoken！請先在 Colab 左側『🔑 Secrets』面板新增名為 'NGROK_AUTHTOKEN' 的金鑰，並勾選開啟允許存取。"

print("設定讀取完成（安全模式）")
print(f"   嵌入模型  : {EMBEDDING_MODEL}")
print(f"   對話模型  : {LLM_MODEL}")
print(f"   Google Drive : {'啟用' if USE_GOOGLE_DRIVE else '停用（記憶體模式）'}")
if USE_GOOGLE_DRIVE:
    print(f"   QA 路徑   : {GDRIVE_QA_PATH}")
    print(f"   向量庫路徑 : {CHROMA_PERSIST_DIR}")

In [ ]:
# ============================================================
# 📦 Cell 3｜安裝系統套件
# ============================================================

print("📦 步驟 1/5：安裝系統工具...")

# 更新 apt 套件清單
!sudo apt-get update -qq 2>&1 | tail -3

# 安裝 pciutils（偵測 GPU）
# 安裝 curl（下載 Ollama 腳本）
# 安裝 zstd（Ollama 壓縮檔解壓必需工具）
!sudo apt-get install -y -qq pciutils curl zstd 2>&1 | tail -3

print("✅ 系統工具安裝完成")

In [ ]:
# ============================================================
# 🦙 Cell 4｜安裝並啟動 Ollama 背景服務
# ============================================================

import subprocess
import threading
import time
import requests

# ── 安裝 Ollama ─────────────────────────────────────────────
print("🦙 步驟 2/5：安裝 Ollama...")

# 改用 Colab 內建感嘆號指令，避免 subprocess 環境變數缺失導致安裝失敗
!curl -fsSL https://ollama.com/install.sh | sh

print("✅ Ollama 安裝指令執行完畢")

# ── 在背景執行緒中啟動 Ollama 服務 ─────────────────────────
def _start_ollama_server():
    """
    在獨立執行緒中啟動 Ollama 推論服務（daemon 模式）。
    使用 daemon=True 確保主程式結束時此執行緒也會自動終止。
    """
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,  # 隱藏標準輸出，避免刷屏
        stderr=subprocess.DEVNULL   # 隱藏錯誤輸出
    )

# 建立並啟動背景執行緒
server_thread = threading.Thread(target=_start_ollama_server, daemon=True)
server_thread.start()

# ── 輪詢等待 Ollama API 就緒（最多 30 秒）──────────────────
print("⏳ 等待 Ollama 服務就緒（最多 30 秒）...")
OLLAMA_API_URL = "http://localhost:11434/api/tags"

for elapsed in range(1, 31):
    try:
        # 設定 (連線超時, 讀取超時)，避免剛啟動時連線成功但讀取逾時報錯
        resp = requests.get(OLLAMA_API_URL, timeout=(2, 3))
        if resp.status_code == 200:
            print(f"✅ Ollama 服務已就緒（等待了 {elapsed} 秒）")
            break
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        # 服務尚未啟動或回應逾時，繼續等待
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        "⏰ Ollama 啟動超時（30秒）！\n"
        "請嘗試：Runtime → Restart session，然後重新執行此 Cell"
    )

# ── 驗證 GPU 是否可用 ────────────────────────────────────────
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                           "--format=csv,noheader"],
                          capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(f"🎮 偵測到 GPU：{gpu_info.stdout.strip()}")
else:
    print("⚠️  未偵測到 GPU，將使用 CPU 推論（速度較慢）")
    print("   建議：Runtime → Change runtime type → T4 GPU")

In [ ]:
# ============================================================
# 🔽 Cell 5｜下載 AI 模型（約 5~15 分鐘）
# ============================================================

print("🔽 步驟 3/5：下載 AI 模型")
print(f"   📐 嵌入模型：{EMBEDDING_MODEL}（約 670MB）")
print(f"   💬 對話模型：{LLM_MODEL}（約 4.1GB）")
print("   ⚠️  首次執行約需 5~15 分鐘，請耐心等候...\n")

# ── 下載嵌入模型 ────────────────────────────────────────────
print(f"⏬ [1/2] 下載嵌入模型 {EMBEDDING_MODEL}...")
!ollama pull {EMBEDDING_MODEL}
print(f"✅ {EMBEDDING_MODEL} 下載完成\n")

# ── 下載對話 LLM 模型 ────────────────────────────────────────
print(f"⏬ [2/2] 下載 LLM 模型 {LLM_MODEL}...")
!ollama pull {LLM_MODEL}
print(f"✅ {LLM_MODEL} 下載完成\n")

# ── 列出已安裝的模型 ─────────────────────────────────────────
print("📋 目前已安裝的模型：")
!ollama list

In [ ]:
# ============================================================
# 📦 Cell 6｜安裝 Python 套件
# ============================================================

print("📦 步驟 4/5：安裝 Python 套件...")

# 安裝所有必要的 Python 套件
# streamlit  : Web UI 框架
# pyngrok    : Ngrok Python 包裝，用於建立公開 HTTPS 隧道
# chromadb   : 向量資料庫
# pandas     : 資料處理（讀取 Excel）
# openpyxl   : pandas 讀取 .xlsx 格式的依賴
# ollama     : Ollama Python SDK
!pip install -q streamlit pyngrok chromadb pandas openpyxl ollama

print("✅ Python 套件安裝完成")

# 顯示關鍵套件版本
import importlib
for pkg in ["streamlit", "chromadb", "ollama", "pyngrok"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"   {pkg}: {mod.__version__}")
    except (ImportError, AttributeError):
        print(f"   {pkg}: 已安裝（版本未知）")

In [ ]:
# ============================================================
# 📊 Cell 7｜準備知識庫資料（自動自 Google Sheets 雲端下載並同步）
# ============================================================

import pandas as pd
import os
import requests

print("📊 準備知識庫資料...")

# ── 選配：掛載 Google Drive ──────────────────────────────────
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    print("💾 掛載 Google Drive...")
    drive.mount("/content/drive", force_remount=False)

    # 實際使用的 QA 檔案路徑與目錄
    qa_file_path = GDRIVE_QA_PATH
    chroma_path  = CHROMA_PERSIST_DIR
    print(f"✅ Google Drive 已掛載")
    print(f"   📁 向量庫路徑    : {chroma_path}")
else:
    qa_file_path = "/content/第二層面試FAQ題庫整併.xlsx"
    chroma_path  = ""
    print("ℹ️  使用記憶體模式（不掛載 Drive）")

# ── 線上同步下載 Excel 檔案（使用使用者提供的新 Google Sheet） ──
sheet_id = "19bO5aD0nbWO5OfjaN0B-jDJd6HCJK24A"
download_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"

try:
    print("📥 正在從雲端下載最新招募小幫手 FAQ 試算表...")
    response = requests.get(download_url)
    if response.status_code == 200:
        # 確保資料夾路徑存在
        os.makedirs(os.path.dirname(qa_file_path), exist_ok=True)
        # 寫入檔案
        with open(qa_file_path, "wb") as f:
            f.write(response.content)
        print(f"✅ 雲端檔案同步成功！本機路徑：{qa_file_path}")
    else:
        print(f"⚠️ 雲端下載失敗（狀態碼 {response.status_code}），將嘗試使用本地既有檔案。")
except Exception as e:
    print(f"⚠️ 雲端下載連線異常：{str(e)}，將嘗試使用本地既有檔案。")

print()

# ── 檢查檔案是否存在 ──
if not os.path.exists(qa_file_path):
    raise FileNotFoundError(
        f"❌ 找不到您的 Excel 知識庫檔案！\n"
        f"請確認以下路徑是否正確，或是否已將檔案放置於 Google Drive：\n"
        f"👉 {qa_file_path}"
    )

# ── 讀取並顯示知識庫預覽 ────────────────────────────────────
try:
    df = pd.read_excel(qa_file_path, header=None)
    print(f"📋 知識庫載入成功：共 {len(df)} 筆資料")
    print("\n前 5 筆資料預覽：")
    for i, row in df.head(5).iterrows():
        col_0 = str(row[0]).strip() if pd.notna(row[0]) else ""
        col_1 = str(row[1]).strip() if len(row) > 1 and pd.notna(row[1]) else ""
        val = f"Q: {col_0} | A: {col_1}"
        preview = val[:80]
        suffix  = "..." if len(val) > 80 else ""
        print(f"  [{i:02d}] {preview}{suffix}")
except Exception as e:
    print(f"❌ 讀取 Excel 檔案時發生錯誤：{str(e)}")
    print("請確認該檔案是否為標準的 Excel (.xlsx) 格式，且內容位於第一個工作表。")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# ✍️ Cell 8｜產生 Streamlit 應用程式檔案 (app.py)
# ============================================================

# 注意：%%writefile 魔法指令必須是 Cell 的第一行
# 以下程式碼將被寫入 /content/app.py
print("✍️ 正在產生 app.py...")

app_code = '''
# -*- coding: utf-8 -*-
"""
LLM + RAG 本地知識問答系統
改寫自 YouTube 教學，適配 Google Colab 部署

技術棧：
  - Streamlit     : Web UI 框架
  - Ollama        : 本地 LLM 推論引擎
  - ChromaDB      : 向量資料庫（語義搜尋）
  - mxbai-embed-large        : 文字嵌入模型
  - ycchen/breeze-7b-instruct-v1_0 : 繁體中文 LLM
"""

import streamlit as st      # Web UI 框架
import ollama               # Ollama Python SDK
import chromadb             # 向量資料庫
import pandas as pd         # 資料處理
import os                   # 系統路徑操作


# ── 常數設定（由 Colab 啟動指令透過環境變數傳入）────────────

# 文字嵌入模型名稱
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "mxbai-embed-large")

# 對話語言模型名稱
LLM_MODEL = os.environ.get("LLM_MODEL", "ycchen/breeze-7b-instruct-v1_0")

# 知識庫 Excel 檔案路徑
QA_FILE_PATH = os.environ.get("QA_FILE_PATH", "/content/QA50.xlsx")

# ChromaDB 持久化路徑（空字串 = 使用記憶體模式）
CHROMA_PERSIST = os.environ.get("CHROMA_PERSIST", "")


def get_chromadb_client():
    """
    根據是否設定持久化路徑，建立並回傳 ChromaDB 客戶端。
    """
    if CHROMA_PERSIST:
        return chromadb.PersistentClient(path=CHROMA_PERSIST)
    else:
        return chromadb.EphemeralClient()


def setup_database():
    """
    初始化向量資料庫，執行步驟如下：
      1. 讀取 Excel 知識庫檔案
      2. 重置既有集合（防止與舊資料混合）
      3. 對每筆資料產生向量並存入 ChromaDB 集合
    """
    client = get_chromadb_client()

    # 讀取 Excel 知識庫
    documents = pd.read_excel(QA_FILE_PATH, header=None)
    total_docs = len(documents)

    # ⚠️ 安全防護：為避免與先前寫入的範例混淆，若集合存在就先刪除重來
    try:
        client.delete_collection(name="recruitment_kb")
    except Exception:
        pass

    # 建立全新的乾淨集合
    collection = client.create_collection(
        name="recruitment_kb",
        metadata={"hnsw:space": "cosine"}
    )

    progress_bar = st.progress(0, text="🔄 正在建立知識庫向量索引...")

    for index, row in documents.iterrows():
        col_0 = str(row[0]).strip() if pd.notna(row[0]) else ""
        col_1 = str(row[1]).strip() if len(row) > 1 and pd.notna(row[1]) else ""

        if not col_0 and not col_1:
            continue

        # 合併問題與回覆細節為單一文件內容
        doc_text = f"【主題/問題】: {col_0}\n【詳細解答/內容】: {col_1}"

        # 呼叫 Ollama 嵌入 API
        response = ollama.embeddings(
            model=EMBEDDING_MODEL,
            prompt=doc_text
        )

        # 將資料寫入向量庫
        collection.add(
            ids=[str(index)],
            embeddings=[response["embedding"]],
            documents=[doc_text]
        )

        progress_pct = (index + 1) / total_docs
        progress_bar.progress(
            progress_pct,
            text=f"🔄 向量索引建立中... ({index + 1}/{total_docs})"
        )

    progress_bar.progress(1.0, text=f"✅ 知識庫索引建立完成！（共 {total_docs} 筆）")

    st.session_state.collection      = collection
    st.session_state.already_executed = True


def initialize():
    """
    檢查 Session State 並按需初始化向量資料庫。
    """
    if "already_executed" not in st.session_state:
        st.session_state.already_executed = False

    if not st.session_state.already_executed:
        with st.spinner("⏳ 首次啟動：正在為您的 Excel 檔案建立知識庫索引..."):
            setup_database()


def handle_user_input(user_input: str, collection) -> None:
    """
    處理使用者問題的完整 RAG 流程
    """
    with st.spinner("🤔 LLM 思考中，請稍候..."):
        # Step 1：取得問題向量
        query_response = ollama.embeddings(
            prompt=user_input,
            model=EMBEDDING_MODEL
        )

        # Step 2：尋找最相似的 3 筆參考資料
        results = collection.query(
            query_embeddings=[query_response["embedding"]],
            n_results=3
        )
        relevant_docs = results["documents"][0]
        context_text  = "\n\n".join(relevant_docs)

        # Step 3：組裝 Prompt 送給 LLM
        prompt = (
            f"以下是從招募知識庫中檢索到的相關內容：\n"
            f"{context_text}\n\n"
            f"請優先根據上述檢索內容，用繁體中文詳細且親切地回答以下問題。如果檢索內容中無直接答案，請基於檢索內容做出合理解釋：\n"
            f"{user_input}"
        )
        llm_output = ollama.generate(
            model=LLM_MODEL,
            prompt=prompt
        )

    # Step 4：渲染回答與來源資訊
    st.markdown("### 💬 回答")
    st.write(llm_output["response"])

    with st.expander("📚 參考的知識片段（點擊展開）", expanded=False):
        for idx, doc in enumerate(relevant_docs, 1):
            st.markdown(f"**[{idx}]** {doc}")
            if idx < len(relevant_docs):
                st.divider()


def main():
    st.set_page_config(
        page_title="招募小幫手 FAQ 系統",
        page_icon="🤖",
        layout="centered",
        initial_sidebar_state="collapsed"
    )

    initialize()

    st.title("🤖 Chatbot 招募小幫手 FAQ 系統")
    st.markdown(
        f"> 📁 檔案來源：`{QA_FILE_PATH.split('/')[-1]}`  \n"
        f"> 🛡️ 核心技術：RAG 語義搜尋 ({EMBEDDING_MODEL}) + 繁中優化 LLM ({LLM_MODEL.split('/')[-1]})"
    )
    st.divider()

    user_input = st.text_area(
        "💬 請輸入您想詢問的招募相關問題：",
        placeholder="例如：實習薪資怎麼算？有什麼福利？履歷該投遞到哪裡？",
        height=120
    )

    col_submit, col_clear, _ = st.columns([1, 1, 3])
    with col_submit:
        submit_btn = st.button("🚀 送出", type="primary", use_container_width=True)
    with col_clear:
        clear_btn  = st.button("🗑️ 清除", use_container_width=True)

    if submit_btn:
        if user_input.strip():
            handle_user_input(user_input, st.session_state.collection)
        else:
            st.warning("⚠️ 請先輸入問題！")

    if clear_btn:
        st.rerun()


if __name__ == "__main__":
    main()
'''

# 將 app.py 寫入 Colab 環境
with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print(f"✅ app.py 產生完成（{len(app_code)} bytes）")
print("   路徑：/content/app.py")

In [ ]:
# ============================================================
# 🚀 Cell 9｜啟動 Streamlit + Ngrok 公開隧道（最後一步）
# ============================================================

import subprocess
import time
import os
from pyngrok import ngrok, conf

print("🚀 步驟 5/5：啟動應用程式...\n")

# ── 傳遞設定給 Streamlit（透過環境變數）────────────────────
# 這樣 app.py 啟動時就能讀到正確的模型名稱和路徑
os.environ["EMBEDDING_MODEL"] = EMBEDDING_MODEL
os.environ["LLM_MODEL"]       = LLM_MODEL
os.environ["QA_FILE_PATH"]    = qa_file_path
os.environ["CHROMA_PERSIST"]  = chroma_path

# ── 設定 Ngrok 認證 token ────────────────────────────────────
conf.get_default().auth_token = NGROK_AUTHTOKEN
print("✅ Ngrok token 設定完成")

# ── 確保沒有殘留的 Streamlit 程序 ─────────────────────────
!pkill -f streamlit 2>/dev/null; echo "舊程序已清理"
time.sleep(2)

# ── 在背景啟動 Streamlit 服務 ────────────────────────────────
# --server.headless=true        : 無瀏覽器模式（Colab 必要）
# --server.fileWatcherType=none : 關閉檔案監視器（Colab 環境穩定性）
# --server.port=8501            : 固定使用 8501 埠
streamlit_proc = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.fileWatcherType=none",
        "--browser.gatherUsageStats=false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ 等待 Streamlit 啟動（5 秒）...")
time.sleep(5)

# ── 確認 Streamlit 程序仍在執行 ────────────────────────────
if streamlit_proc.poll() is not None:
    # 程序已結束，表示啟動失敗
    stdout, stderr = streamlit_proc.communicate()
    raise RuntimeError(
        f"❌ Streamlit 啟動失敗！\n"
        f"錯誤訊息：{stderr.decode('utf-8', errors='replace')}"
    )

print("✅ Streamlit 服務已啟動（port 8501）")

# ── 建立 Ngrok 公開隧道 ──────────────────────────────────────
# 將 Colab 內部的 localhost:8501 映射為對外的 HTTPS URL
print("🌐 建立 Ngrok 公開隧道...")
tunnel = ngrok.connect(8501)
public_url = tunnel.public_url

# ── 顯示結果 ─────────────────────────────────────────────────
print()
print("=" * 62)
print(f"  🌐 應用程式公開網址：")
print(f"  👉 {public_url}")
print("=" * 62)
print()
print("📌 注意事項：")
print("  • 此連結在本次 Colab Session 期間有效")
print("  • Runtime 斷線後連結失效，需重新執行 Cell 4 → Cell 9")
print("  • 若需停止：執行 ngrok.kill() 並重啟 Runtime")
print()
print("🎉 首次開啟連結時，Streamlit 需要幾秒鐘載入，請稍候！")

### 🧪 額外測試：在 Colab 頁面直接進行本地 RAG 問答測試 (無需 Streamlit/Ngrok)
執行下方單元格後，您可以直接在輸入欄位輸入問題，系統會自動在向量資料庫搜尋最相似的知識，並由本地 `Breeze-7B` 模型回答您。

In [ ]:
# ============================================================
# 🧪 Colab 本地直接問答測試單元（不依賴 Web UI）- 3欄位精準 RAG 優化版
# ============================================================

import os
import subprocess
import threading
import time
import requests
import ollama
import chromadb
import pandas as pd
from google.colab import drive

def ensure_ollama_running():
    """確保 Ollama 伺服器正在背景執行"""
    print("🦙 正在檢查背景 Ollama 服務狀態...")
    url = "http://localhost:11434/api/tags"
    try:
        resp = requests.get(url, timeout=2)
        if resp.status_code == 200:
            print("✅ Ollama 服務正常運作中")
            return
    except Exception:
        pass

    print("⚠️  偵測到 Ollama 未啟動，正在背景主動重啟服務...")
    def _start_server():
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    t = threading.Thread(target=_start_server, daemon=True)
    t.start()

    # 等待服務就緒
    for idx in range(1, 16):
        try:
            resp = requests.get(url, timeout=2)
            if resp.status_code == 200:
                print(f"✅ Ollama 服務已成功啟動！（等待了 {idx} 秒）")
                return
        except Exception:
            pass
        time.sleep(1)
    print("❌ 無法成功啟動 Ollama 服務，請確認 Cell 4 安裝程序是否正常。")

def init_and_query():
    # 1. 確保 Drive 有掛載
    print("💾 正在確認 Google Drive 狀態...")
    drive.mount("/content/drive", force_remount=False)

    # 2. 確保 Ollama 背景執行
    ensure_ollama_running()

    # 3. 設定參數
    embedding_model = "mxbai-embed-large"
    llm_model = "ycchen/breeze-7b-instruct-v1_0"
    gdrive_qa_path = "/content/drive/MyDrive/ChatbotRAG/第二層面試FAQ題庫整併.xlsx"
    chroma_persist_dir = "/content/drive/MyDrive/ChatbotRAG/chroma_db_advanced"

    # 4. 連接 ChromaDB
    print("📂 讀取向量資料庫...")
    client = chromadb.PersistentClient(path=chroma_persist_dir)

    # 為了套用最精準的 3 欄位對應（分類、問法、回覆），強制重新建庫，避免舊有殘缺資料干擾
    collection_name = "recruitment_kb_advanced"
    try:
        client.delete_collection(name=collection_name)
        print("🧹 已清理舊有向量快取，準備建立全新 3 欄位對照資料...")
    except Exception:
        pass

    print("⚠️ 正在讀取 Excel FAQ 進行 3 欄位精準化向量索引建立...")
    if not os.path.exists(gdrive_qa_path):
        print(f"❌ 找不到 FAQ Excel 檔案，路徑為：{gdrive_qa_path}")
        return

    # 讀取 Excel 檔案
    df_qa = pd.read_excel(gdrive_qa_path, header=None)
    collection = client.create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})

    count = 0
    for index, row in df_qa.iterrows():
        # 讀取第 0 欄（分類）、第 1 欄（問題問法）、第 2 欄（回覆內容）
        col_0 = str(row[0]).strip() if pd.notna(row[0]) else ""
        col_1 = str(row[1]).strip() if len(row) > 1 and pd.notna(row[1]) else ""
        col_2 = str(row[2]).strip() if len(row) > 2 and pd.notna(row[2]) else ""

        # 跳過 Excel 的標頭文字行（第一行）
        if "題目分類" in col_0 or "題庫問法" in col_1:
            continue

        if not col_0 and not col_1 and not col_2:
            continue

        # 合併三欄位資訊，作為 RAG 精準比對與生成的背景知識
        doc_text = f"【類別】: {col_0}\n【問題】: {col_1}\n【官方解答】: {col_2}"

        # 生成 Embedding 並寫入向量庫
        response = ollama.embeddings(model=embedding_model, prompt=doc_text)
        collection.add(
            ids=[f"row_{index}"],
            embeddings=[response["embedding"]],
            documents=[doc_text]
        )
        count += 1
    print(f"✅ 進階知識庫向量索引重新建立完畢！共導入 {count} 筆包含官方解答的完整對話資訊。")

    # 5. 進入多輪問答無限迴圈
    print("\n🚀 多輪本地問答測試系統已準備就緒！")
    print("💡 提示：輸入 'exit'、'quit' 或直接按 Enter 鍵可結束測試。")

    while True:
        print("\n" + "="*60)
        user_question = input("💬 [提問] 請輸入您的問題： ").strip()

        # 結束語意判斷
        if not user_question or user_question.lower() in ["exit", "quit", "離開", "結束"]:
            print("\n👋 已結束多輪問答測試，感謝您的使用！")
            break

        print("\n🔍 正在檢索最相關的本地 FAQ 片段...")
        try:
            # 取得問題 Embedding
            query_emb = ollama.embeddings(model=embedding_model, prompt=user_question)["embedding"]
            # 搜尋向量庫
            results = collection.query(query_embeddings=[query_emb], n_results=3)
            relevant_docs = results["documents"][0]
            context_text = "\n\n".join(relevant_docs)

            print("🤔 正在使用 Breeze-7B 生成回答，請稍候...")
            prompt = (
                f"以下是從招募知識庫中檢索到的相關內容：\n"
                f"{context_text}\n\n"
                f"請優先根據上述檢索內容，用繁體中文詳細且親切地回答以下問題。如果檢索內容中無直接答案，請基於檢索內容做出合理解釋：\n"
                f"{user_question}"
            )

            llm_output = ollama.generate(model=llm_model, prompt=prompt)

            print("\n🤖 系統回答：")
            print(llm_output["response"])

            print("\n📚 參考知識片段：")
            for idx, doc in enumerate(relevant_docs, 1):
                # 將換行排版稍微縮排整理，方便在終端機閱讀
                clean_doc = doc.replace('\n', '\n    ')
                print(f"  👉 參考來源 [{idx}]:\n    {clean_doc}")

        except Exception as e:
            print(f"❌ 處理問答時發生錯誤：{str(e)}")

# 執行對話
init_and_query()

💾 正在確認 Google Drive 狀態...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🦙 正在檢查背景 Ollama 服務狀態...
⚠️  偵測到 Ollama 未啟動，正在背景主動重啟服務...
✅ Ollama 服務已成功啟動！（等待了 2 秒）
📂 讀取向量資料庫...
🧹 已清理舊有向量快取，準備建立全新 3 欄位對照資料...
⚠️ 正在讀取 Excel FAQ 進行 3 欄位精準化向量索引建立...
✅ 進階知識庫向量索引重新建立完畢！共導入 46 筆包含官方解答的完整對話資訊。

🚀 多輪本地問答測試系統已準備就緒！
💡 提示：輸入 'exit'、'quit' 或直接按 Enter 鍵可結束測試。

💬 [提問] 請輸入您的問題： 麥當勞月薪?大夜班薪資?

🔍 正在檢索最相關的本地 FAQ 片段...
🤔 正在使用 Breeze-7B 生成回答，請稍候...

🤖 系統回答：
在麥當勞的薪資制度中，正職服務員（正職人員）的月薪通常從新台幣32,000元起（大夜正職的部分，則從41,000元起）。而大夜班的薪資方面，時薪為251元起。

📚 參考知識片段：
  👉 參考來源 [1]:
    【類別】: 薪資與收入
    【問題】: 薪水什麼時候發？
    【官方解答】: 麥當勞的發薪制度為每個月發薪兩次（雙週發薪）。
  👉 參考來源 [2]:
    【類別】: 薪資與收入
    【問題】: 一個月大概可以領多少？
    【官方解答】: 回答前請先確認您應徵或詢問的職務（例如：計時人員、正職服務員或餐廳儲備幹部）：  - 計時人員：依實際排班時數計算，一般時薪 196 元起，大夜時薪 251 元起
      - 正職服務員：月薪 32,000 元起（大夜正職 41,000 元起）
      - 餐廳儲備幹部：月薪 41,000 元起
  👉 參考來源 [3]:
    【類別】: 工時與班別
    【問題】: 一個月最低工時是多少？
    【官方解答】: 麥當勞工時規範依職務而定：  - 計時人員：無單月最低總